In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent    # notebooks -> project_root
sys.path.insert(0, str(project_root))
print(project_root)

from matplotlib import pyplot as plt
import numpy as np

/Users/tcui0786/Work/DeepTransport/PyFInS-public


In [2]:
from NumpyModels.RandomModel import RandomModel
from NumpyModels import GaussianDiagonalPrior
from BaseModels import GaussianLikelihood
from NumpyModels.DerivativeTests import test_combined

m = 4
n = 4
sig2noise = 10.0

prior = GaussianDiagonalPrior(dim=n, mu=0.0, decay_rate=2.0, variance=4.0)

# True parameter
true_u = prior.sample(1)
model = RandomModel(m, n, nonlinear=True)
y, _ = model.solve(true_u)

sigma = np.abs(y).mean()/sig2noise
fake_data = y + np.random.normal(0, sigma, size=y.shape)
like = GaussianLikelihood(fake_data, sigma)

test_combined(prior, model, n_tests=50)

Solver and prior derivative tests
  number of tests : 50
  epsilon         : 1.000e-06

test                             mean relative error    max relative error
--------------------------------------------------------------------------
prior L/L.T adjoint                     2.982169e-17          2.477316e-16
prior inverse                           1.340909e-17          1.366938e-16
physical Jacobian                       2.531122e-10          1.391413e-09
physical adjoint                        2.629404e-16          1.958603e-15
whitened Jacobian                       3.183563e-10          2.792165e-09
whitened adjoint                        9.470596e-16          3.465913e-14


In [3]:
# Only for running SMC 
from NumpyInference import SMC 
from InferenceTools.TransitionKernels import OneStepKernel

kernel = OneStepKernel(
        type="OW",
        sigma=-3.0,
        adapt=True,
        nbatch=1,
)

sampler = SMC(
        prior,
        model,
        like,
        kernel,
        max_iter=100,
        n_samples=1000,
        n_mutation_steps=20,
        n_final_mutation_steps=100,
        ess_tol=0.5,
        min_beta=1e-6,
)

sampler.iterate()

iteration 0
temperature 0.009629156054740566, ess 0.4917645106389941, log const -1.0947980974472593


MCMC: 100%|██████████| 20/20 [00:00<00:00, 90.83it/s, acc_rate=0.776, sigma=-2.00000] 

iteration 1
temperature 0.0677891499326604, ess 0.49795735003615216, log const -1.2081164680687226


MCMC: 100%|██████████| 20/20 [00:00<00:00, 51.41it/s, acc_rate=0.494, sigma=-1.80000]

iteration 2
temperature 0.3230113316355213, ess 0.4973557893774138, log const -1.3872712217945433


MCMC: 100%|██████████| 20/20 [00:00<00:00, 62.84it/s, acc_rate=0.413, sigma=-2.80000]

iteration 3
temperature 1.0, ess 0.5366249940408134, log const -1.4449066733831368


MCMC: 100%|██████████| 100/100 [00:01<00:00, 56.47it/s, acc_rate=0.491, sigma=-4.00000]
